In [9]:
import pandas as pd
import joblib

from utils import fit_position_pipeline, select_features_by_correlation

In [10]:
features = joblib.load('../predictors/hist/feats_gk')
features[6]

['was_home',
 'transfers_in',
 'transfers_out',
 'difficulty',
 'opponent_difficulty',
 'xP',
 'elo_diff',
 'yellow_cards_rolling_3',
 'expected_goal_involvements_rolling_1',
 'expected_goal_involvements_rolling_3',
 'tackles_rolling_3',
 'interceptions_per_90_rolling_1',
 'interceptions_per_90_rolling_3',
 'chances_created_per_90_rolling_3',
 'sweeper_actions_per_90_rolling_3',
 'clean_sheets_1_ewm',
 'expected_goals_conceded_1_ewm',
 'tackles_1_ewm',
 'goals_prevented_3_ewm',
 'opponent_xG_5_ewm',
 'big_chances_faced_5_ewm']

In [12]:
# features = joblib.load('../predictors/hist/feats_gk')
features =  [
        'was_home', 'difficulty', 'opponent_difficulty', 'win_prob', 'percentage_net_transfers', 'elo_diff', 'points_5_ewm', 'xP_5_ewm', 'minutes_rolling_3',
        'clean_sheets_3_ewm', 'goals_conceded_3_ewm', 'yellow_cards_rolling_5', 'saves_rolling_5', 'ict_index_rolling_3', 'starts_rolling_3',
        'shots_faced_3_ewm', 'points_3_ewm', 'saves_3_ewm', 'ict_index_3_ewm', 'expected_goals_conceded_3_ewm', 'goals_prevented_3_ewm', 'opponent_xG_3_ewm',
]


player_data = pd.read_csv('../rolled_data_24_25.csv')
player_data = player_data[player_data['position'] == 'Goalkeeper']

league_models = {}

for gw in range(6,39):
    print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
    data = player_data[(player_data['round']>3) & (player_data['round'] < gw)].fillna(0)
    feats = features #features[gw]

    # feats.remove('xP')
    # print('--------', feats)
    selected_feats = select_features_by_correlation(data, feats, target_col='xP', redundancy_threshold=0.7, min_corr=0.1, verbose=True)

    league_model = fit_position_pipeline(
            data, selected_feats, gw, position_name='GK', random_seed=42
        )

    print(f"Features for GK kept in gw {gw} -> {league_model['feats']}")

    league_models[gw] = league_model




 Model for round 6 
  dropping 'points_5_ewm' (|corr| with xP=0.895) -- |corr|=0.974 with already-selected 'points_3_ewm'
  dropping 'minutes_rolling_3' (|corr| with xP=0.810) -- |corr|=0.864 with already-selected 'points_3_ewm'
  dropping 'starts_rolling_3' (|corr| with xP=0.809) -- |corr|=0.861 with already-selected 'points_3_ewm'
  dropping 'xP_5_ewm' (|corr| with xP=0.794) -- |corr|=0.821 with already-selected 'points_3_ewm'
  dropping 'ict_index_3_ewm' (|corr| with xP=0.759) -- |corr|=0.867 with already-selected 'points_3_ewm'
  dropping 'clean_sheets_3_ewm' (|corr| with xP=0.757) -- |corr|=0.825 with already-selected 'points_3_ewm'
  dropping 'ict_index_rolling_3' (|corr| with xP=0.739) -- |corr|=0.832 with already-selected 'points_3_ewm'
  dropping 'saves_3_ewm' (|corr| with xP=0.714) -- |corr|=0.821 with already-selected 'points_3_ewm'
  dropping 'saves_rolling_5' (|corr| with xP=0.690) -- |corr|=0.778 with already-selected 'points_3_ewm'
  dropping 'expected_goals_conceded_3_

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

ValueError: Not enough samples to build a trace.

In [ ]:
joblib.dump(league_models, './estimates/league_models_gk')